In [2]:
import pandas as pd

df = pd.read_csv("data/DataCoSupplyChainDataset.csv", encoding="latin-1")

print(df.shape)

(180519, 53)


In [3]:
print(df.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrde

In [4]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64


In [5]:
cols_to_drop = [
    "Customer Email", "Customer Password", "Customer Fname",
    "Customer Lname", "Customer Street", "Customer Zipcode",
    "Product Description", "Product Image", "Order Zipcode"
]

df = df.drop(columns=cols_to_drop)

print(df.shape)

(180519, 44)


In [6]:
print(df.duplicated().sum())
print(df["Order Item Id"].is_unique)

0
True


In [7]:
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping date (DateOrders)"] = pd.to_datetime(df["shipping date (DateOrders)"])

print(df["order date (DateOrders)"].min())
print(df["order date (DateOrders)"].max())

2015-01-01 00:00:00
2018-01-31 23:38:00


In [8]:
print(df.groupby("Shipping Mode")["Late_delivery_risk"].mean().round(3))
print(df.groupby("Shipping Mode")["Days for shipment (scheduled)"].unique())

Shipping Mode
First Class       0.953
Same Day          0.457
Second Class      0.766
Standard Class    0.381
Name: Late_delivery_risk, dtype: float64
Shipping Mode
First Class       [1]
Same Day          [0]
Second Class      [2]
Standard Class    [4]
Name: Days for shipment (scheduled), dtype: object


In [9]:
print(df.groupby("Shipping Mode")["Days for shipping (real)"].agg(["mean", "min", "max"]).round(2))

                mean  min  max
Shipping Mode                 
First Class     2.00    2    2
Same Day        0.48    0    1
Second Class    3.99    2    6
Standard Class  4.00    2    6


In [10]:
print(pd.crosstab(df["Shipping Mode"], df["Delivery Status"]))

Delivery Status  Advance shipping  Late delivery  Shipping canceled  \
Shipping Mode                                                         
First Class                     0          26513               1301   
Same Day                        0           4454                444   
Second Class                    0          26987               1410   
Standard Class              41592          41023               4599   

Delivery Status  Shipping on time  
Shipping Mode                      
First Class                     0  
Same Day                     4839  
Second Class                 6819  
Standard Class              20538  


In [11]:
shipped = df[df["Delivery Status"] != "Shipping canceled"]

print(shipped.shape)
print(shipped.groupby("Shipping Mode")["Late_delivery_risk"].mean().round(3))

(172765, 44)
Shipping Mode
First Class       1.000
Same Day          0.479
Second Class      0.798
Standard Class    0.398
Name: Late_delivery_risk, dtype: float64


In [12]:
print((df["Benefit per order"] == df["Order Profit Per Order"]).mean())
print((df["Product Price"] == df["Order Item Product Price"]).mean())
print((df["Sales per customer"] == df["Order Item Total"]).mean())

1.0
1.0
1.0


In [13]:
df = df.drop(columns=["Benefit per order", "Product Price", "Sales per customer"])
print(df.shape)

print((df["Category Id"] == df["Product Category Id"]).mean())
print((df["Customer Id"] == df["Order Customer Id"]).mean())
print((df["Order Item Cardprod Id"] == df["Product Card Id"]).mean())

(180519, 41)
1.0
1.0
1.0


In [14]:
df = df.drop(columns=["Product Category Id", "Order Customer Id", "Order Item Cardprod Id"])

shipped = df[df["Delivery Status"] != "Shipping canceled"]

print(df.shape)
print(shipped.shape)

(180519, 38)
(172765, 38)


In [15]:
print((df["Order Profit Per Order"] < 0).mean().round(3))
print(df["Order Profit Per Order"].describe().round(2))
print((df["Sales"] <= 0).sum())

0.187
count    180519.00
mean         21.97
std         104.43
min       -4274.98
25%           7.00
50%          31.52
75%          64.80
max         911.80
Name: Order Profit Per Order, dtype: float64
0


In [16]:
df.to_csv("data/dataco_clean.csv", index=False)
print("Saved")

Saved


In [17]:
%pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB 660.6 kB/s eta 0:00:20
   ---------------------------------------- 0.1/13.2 MB 1.3 MB/s eta 0:00:10
   - -------------------------------------- 0.5/13.2 MB 4.2 MB/s eta 0:00:04
   --- ------------------------------------ 1.2/13.2 MB 8.4 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/13.2 MB 11.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.8/13.2 MB 12.9 MB/s eta 0:00:01
   ----------- ---------------------------- 3.6/13.2 MB 14.5 MB/s eta 0:00:01
   ------------- -------------------------- 4.5/13.2 MB 16.0 MB/s eta 0:00:01
   ---------------- ----------------------- 5.5/13.2 MB 16.7 MB/s eta 0:00:01
   ------------------- -------------------- 6.4/13.2 MB 17.7 MB/s eta 0:00:01
   -


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
